# Evaluation on unseen dataset

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import sys
from sklearn.metrics import classification_report

sys.path.append('../')
from config import *

In [2]:
df_test = pd.read_csv(RAW_DIR / "UNSW_NB15_testing-set.csv")

print(f"Test set shape: {df_test.shape}")
print(f"Label distribution:\n{df_test['label'].value_counts()}")

Test set shape: (82332, 45)
Label distribution:
label
1    45332
0    37000
Name: count, dtype: int64


## 1. Data processing

In [3]:
selected_features = ['dur','sbytes','dbytes','rate','sttl','dttl','sinpkt','dinpkt','smean','dmean','proto','service','state','label']

df_test = df_test[selected_features]
df_test.head()

,dur,sbytes,dbytes,rate,sttl,dttl,sinpkt,dinpkt,smean,dmean,proto,service,state,label
0,0.000011,496,0,90909.0902,254,0,0.011,0.0,248,0,udp,-,INT,0
1,0.000008,1762,0,125000.0003,254,0,0.008,0.0,881,0,udp,-,INT,0
2,0.000005,1068,0,200000.0051,254,0,0.005,0.0,534,0,udp,-,INT,0
3,0.000006,900,0,166666.6608,254,0,0.006,0.0,450,0,udp,-,INT,0
4,0.000010,2126,0,100000.0025,254,0,0.010,0.0,1063,0,udp,-,INT,0


In [4]:
# Log transforms
for col in ['rate','sbytes','dbytes','sinpkt','dinpkt','dmean']:
    df_test[col] = np.log1p(df_test[col])

for col in ['dur','smean']:
    df_test[col] = np.log1p(df_test[col])
    df_test[col] = np.log1p(df_test[col])

In [5]:
# categorical feature analysis
print(f"Number of unique protocols: {df_test['proto'].nunique()}")
print(f"Number of unique services:  {df_test['service'].nunique()}")
print(f"Number of unique states:    {df_test['state'].nunique()}")

Number of unique protocols: 131
Number of unique services:  13
Number of unique states:    7


In [6]:
df_ohe = df_test.copy()
df_ohe = pd.get_dummies(df_ohe, columns=['service', 'state'], drop_first=False)

print(f"Shape of the ohe test set: {df_ohe.shape}")

Shape of the ohe test set: (82332, 32)


In [7]:
df_ohe.head()

,dur,sbytes,dbytes,rate,sttl,dttl,sinpkt,dinpkt,smean,dmean,...,service_snmp,service_ssh,service_ssl,state_ACC,state_CLO,state_CON,state_FIN,state_INT,state_REQ,state_RST
0,0.000011,6.208590,0.0,11.417626,254,0,0.010940,0.0,1.874484,0.0,...,False,False,False,False,False,False,False,True,False,False
1,0.000008,7.474772,0.0,11.736077,254,0,0.007968,0.0,2.051838,0.0,...,False,False,False,False,False,False,False,True,False,False
2,0.000005,6.974479,0.0,12.206078,254,0,0.004988,0.0,1.985442,0.0,...,False,False,False,False,False,False,False,True,False,False
3,0.000006,6.803505,0.0,12.023757,254,0,0.005982,0.0,1.961709,0.0,...,False,False,False,False,False,False,False,True,False,False
4,0.000010,7.662468,0.0,11.512935,254,0,0.009950,0.0,2.075658,0.0,...,False,False,False,False,False,False,False,True,False,False


### 1.1 Aligning one hot encoded columns

In [8]:
with open(PROCESSED_DIR / 'ohe_columns.json', 'r') as f:
    train_ohe_cols = json.load(f)

missing = [col for col in train_ohe_cols if col not in df_ohe.columns]
# ['state_ECO', 'state_PAR', 'state_URN', 'state_no']

for col in missing:
    df_ohe[col] = 0

extra = [col for col in df_ohe.columns if col not in train_ohe_cols + ['label']]
# ['state_ACC', 'state_CLO']

df_ohe = df_ohe.drop(columns=extra)

df_ohe = df_ohe[train_ohe_cols + ['label']]

print(f"Missing columns added: {len(missing)}")
print(f"Extra columns removed: {len(extra)}")
print(f"Test OHE shape: {df_ohe.shape}")

Missing columns added: 4
Extra columns removed: 2
Test OHE shape: (82332, 34)


### 1.2 Windowing

In [9]:
# Aggregation dictionary for windowing
agg_dict = {}

continuous_cols = ['dur','sbytes','dbytes','rate','sttl',
                   'dttl','sinpkt','dinpkt','smean','dmean']

for col in continuous_cols:
    agg_dict[col] = ['mean','std']

one_hot_cols = [col for col in df_ohe.columns if col.startswith('service') or col.startswith('state')]

for col in one_hot_cols:
    agg_dict[col] = 'mean'

agg_dict['proto'] = 'nunique'
agg_dict['label'] = 'max'

df_ohe['window_id'] = df_ohe.index // WINDOW_SIZE

df_windowed = df_ohe.groupby('window_id').agg(agg_dict)
df_windowed.columns = ['_'.join(col).strip() for col in df_windowed.columns]

df_windowed = df_windowed.rename(columns={'label_max': 'window_attack'})
df_windowed = df_windowed.reset_index()

### 1.3 Aligning to final expected columns

In [10]:
with open(PROCESSED_DIR / 'windowed_columns.json', 'r') as f:
    expected_cols = json.load(f) # train_windowed_cols

extra = [col for col in df_windowed.columns if col not in expected_cols]
df_windowed.drop(columns=extra, inplace=True)
# ['state_PAR_mean', 'state_URN_mean', 'state_no_mean']

if list(df_windowed.columns) == expected_cols:
    print(f"Columns aligned") 

Columns aligned


In [11]:
print("=== Test Data ===")
print(f"Shape: {df_windowed.shape}")
print(f"\n{df_windowed['window_attack'].value_counts()}")

=== Test Data ===
Shape: (824, 42)

window_attack
1    455
0    369
Name: count, dtype: int64


## 2. Testing

In [12]:
# Model loading
scaler = joblib.load(MODEL_DIR / 'scaler.pkl')
iso = joblib.load(MODEL_DIR / 'isolation_forest.pkl')
rf_stage2 = joblib.load(MODEL_DIR / 'rf_stage2.pkl')

# Scaling
feature_cols = [col for col in df_windowed.columns if col not in ['window_id', 'window_attack']]
X_test = df_windowed[feature_cols]
X_test_scaled = scaler.transform(X_test)

### 2.1 Stage 1 - Isolation Forest

In [13]:
df_windowed['anomaly_score'] = iso.decision_function(X_test_scaled)
df_windowed['stage1_pred'] = (df_windowed['anomaly_score'] < BEST_THRESHOLD).astype(int)

print(f"Stage 1 (IF) windows flagged: {df_windowed['stage1_pred'].sum()}")
print(f"Actual attacks: {df_windowed['window_attack'].sum()}")

Stage 1 (IF) windows flagged: 559
Actual attacks: 455


### 2.2 Stage 2 - Random Forest on flagged windows

In [14]:
flagged_test = df_windowed[df_windowed['stage1_pred'] == 1].copy()

X_flagged = flagged_test[feature_cols]
flagged_test['stage2_pred'] = rf_stage2.predict(X_flagged)

df_windowed['final_pred'] = df_windowed['stage1_pred'].copy()

df_windowed.loc[flagged_test.index, 'final_pred'] = flagged_test['stage2_pred']
df_windowed.to_csv(PROCESSED_DIR / 'test_windowed_results.csv', index=False)

print(f"Stage 2 downgraded: {(flagged_test['stage2_pred'] == 0).sum()} windows")
print(f"Final flagged: {df_windowed['final_pred'].sum()} windows")

Stage 2 downgraded: 102 windows
Final flagged: 457 windows


## 3. Summary

In [15]:
print("=== Stage 1 (IF only) ===")
print(classification_report(df_windowed['window_attack'], df_windowed['stage1_pred']))

print("=== Two-Stage (IF + RF) ===")
print(classification_report(df_windowed['window_attack'], df_windowed['final_pred']))

for name, pred_col in [('Stage 1 IF', 'stage1_pred'), 
                       ('Two-Stage', 'final_pred')]:
    fp = ((df_windowed['window_attack']==0) & (df_windowed[pred_col]==1)).sum()
    fn = ((df_windowed['window_attack']==1) & (df_windowed[pred_col]==0)).sum()
    
    print(f"{name:14s} → FP: {fp:2d}, FN: {fn:1d}")

=== Stage 1 (IF only) ===
              precision    recall  f1-score   support

           0       0.96      0.69      0.80       369
           1       0.80      0.98      0.88       455

    accuracy                           0.85       824
   macro avg       0.88      0.83      0.84       824
weighted avg       0.87      0.85      0.84       824

=== Two-Stage (IF + RF) ===
              precision    recall  f1-score   support

           0       0.97      0.96      0.97       369
           1       0.97      0.98      0.97       455

    accuracy                           0.97       824
   macro avg       0.97      0.97      0.97       824
weighted avg       0.97      0.97      0.97       824

Stage 1 IF     → FP: 114, FN: 10
Two-Stage      → FP: 13, FN: 11
